# 🗣️ Design Virgo's voice — Virgo-1.0-Angkor-Voice (Kaggle or Colab, free GPU)

Makes **Virgo's own voice** with **VoxCPM2** (OpenBMB, **Apache 2.0**: free for commercial use; speaks **Khmer**, English and 28 more languages). You describe a voice in words, listen to a few versions, pick the one you like, and it's saved privately to Hugging Face as **Virgo-1.0-Angkor-Voice**. The Virgo server notebooks then use it automatically for chat and Virgo Live.

The voice is made from a *description*, so it's Virgo's own — not a copy of Gemini's or anyone's voice. (Only use a real person's recordings with their written permission.)

**Run**
1. **Kaggle**: Session options → **GPU T4** and **Internet On**; Add-ons → Secrets: `HF_TOKEN` (write access), ticked for this notebook. **Colab**: Runtime → **T4 GPU**; 🔑 Secrets: `HF_TOKEN` with notebook access.
2. Run cells 1–2 and **listen** to the voices. Change the descriptions and run cell 2 again until you love one.
3. Set `CHOICE` in cell 3 and run it: the voice is uploaded.
4. *(Optional)* Cell 4 fine-tunes a LoRA so the voice is steadier and a bit faster (about 1–2 hours).

In [ ]:
# ⚙️ Settings: describe the voice (English works best for descriptions). Try a few!
DESCRIPTIONS = [
    "A young woman, warm and friendly, bright clear voice, calm natural pace",
    "A young woman, gentle and cheerful, soft smile in the voice, clear pronunciation",
    "A woman in her late twenties, confident and kind, smooth clear voice, relaxed pace",
    "A young man, warm and calm, clear friendly voice, natural pace",
]
SEEDS = [7, 21]  # each description is tried with these seeds (different takes)
# The sample Virgo speaks: about 8–12 seconds. The chosen take becomes Virgo's voice sample.
SAMPLE_KM = "សួស្តី! ខ្ញុំជា Virgo AI ជំនួយការរបស់អ្នក។ ថ្ងៃនេះ តើខ្ញុំអាចជួយអ្វីបានខ្លះ? ខ្ញុំអាចឆ្លើយសំណួរ បកប្រែ និងជួយរៀបចំការងាររបស់អ្នក។"
SAMPLE_EN = "Hello! I'm Virgo AI. How can I help you today?"
MODEL_NAME = "Virgo-1.0-Angkor-Voice"

In [ ]:
# 1️⃣ Set up (VoxCPM2 and your Hugging Face login)
import os, subprocess, sys, torch, json
assert torch.cuda.is_available(), "❌ No GPU: turn on a T4 GPU (see the steps above)"
# The newest VoxCPM from GitHub (matches its training scripts); the pip release if GitHub can't be reached.
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/OpenBMB/VoxCPM"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "voxcpm"], check=True)
def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
from huggingface_hub import login, HfApi, snapshot_download
login(token=secret("HF_TOKEN") or input("Hugging Face token (hf_..., write access): ").strip())
ME = HfApi().whoami()["name"]
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
BASE = snapshot_download("openbmb/VoxCPM2")
if torch.cuda.get_device_capability(0)[0] < 8:  # T4: no real bfloat16, so run in float16 (much faster)
    cfg_path = os.path.join(BASE, "config.json"); cfg = json.load(open(cfg_path))
    if cfg.get("dtype") != "float16":
        cfg["dtype"] = "float16"; text = json.dumps(cfg, indent=2)
        os.remove(cfg_path); open(cfg_path, "w").write(text)  # replace the cache link, don't write through it
from voxcpm import VoxCPM
model = VoxCPM.from_pretrained(BASE, load_denoiser=False)
RATE = model.tts_model.sample_rate
import inspect, random, numpy as np
_GEN_ARGS = set(inspect.signature(model._generate).parameters)
def speak(text, seed=7, **kw):
    """model.generate that works on every VoxCPM version (older ones have no `seed` argument)."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if "seed" in _GEN_ARGS:
        kw["seed"] = seed
    return model.generate(text=text, **{k: v for k, v in kw.items() if k in _GEN_ARGS})
print("✅ VoxCPM2 ready on", torch.cuda.get_device_name(0), "| saving as", f"{ME}/{MODEL_NAME}")

In [ ]:
# 2️⃣ Listen: every description × seed, in Khmer and English
import soundfile as sf, time
from IPython.display import Audio, display, Markdown
os.makedirs(f"{WORK}/voices", exist_ok=True)
takes = []
for d in DESCRIPTIONS:
    for seed in SEEDS:
        n = len(takes) + 1
        start = time.time()
        km = speak(f"({d}){SAMPLE_KM}", cfg_value=2.0, inference_timesteps=10, seed=seed)
        en = speak(f"({d}){SAMPLE_EN}", cfg_value=2.0, inference_timesteps=10, seed=seed)
        sf.write(f"{WORK}/voices/{n}_km.wav", km, RATE); sf.write(f"{WORK}/voices/{n}_en.wav", en, RATE)
        takes.append({"description": d, "seed": seed})
        took = time.time() - start; spoken = (len(km) + len(en)) / RATE
        display(Markdown(f"**{n}.** {d} · seed {seed} · made {spoken:.0f} s of speech in {took:.0f} s"))
        display(Audio(km, rate=RATE)); display(Audio(en, rate=RATE))

In [ ]:
# 3️⃣ Save your favourite as Virgo's voice (uploaded privately to Hugging Face)
CHOICE = 1  # the number of the take you liked
import shutil
take = takes[CHOICE - 1]
out = f"{WORK}/{MODEL_NAME}"; os.makedirs(out, exist_ok=True)
shutil.copy(f"{WORK}/voices/{CHOICE}_km.wav", f"{out}/voice.wav")
json.dump({**take, "text": SAMPLE_KM, "base": "openbmb/VoxCPM2"}, open(f"{out}/voice.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
open(f"{out}/README.md", "w").write(f"""---
license: apache-2.0
base_model: openbmb/VoxCPM2
language: [km, en]
---
# {MODEL_NAME}

Virgo AI's voice (by KSN), built on [VoxCPM2](https://huggingface.co/openbmb/VoxCPM2) (Apache 2.0).
Designed from the description: *{take['description']}*.
""")
repo = f"{ME}/{MODEL_NAME}"
HfApi().create_repo(repo, private=True, exist_ok=True)
HfApi().upload_folder(folder_path=out, repo_id=repo, commit_message=f"Virgo's voice: take {CHOICE}")
# Check: Virgo says something new in the saved voice
check = speak("ខ្ញុំរីករាយណាស់ដែលបានជួបអ្នក។ Nice to meet you!", prompt_wav_path=f"{out}/voice.wav",
                       prompt_text=SAMPLE_KM, reference_wav_path=f"{out}/voice.wav", cfg_value=2.0, inference_timesteps=10, seed=7)
display(Audio(check, rate=RATE))
print("☁️ Saved to", repo, "— restart the Virgo server notebook and Virgo speaks with this voice.")

### 4️⃣ Optional: fine-tune a LoRA (steadier voice, better Khmer)
Trains on sentences spoken in your chosen voice — or, better, on **real Khmer recordings** from a speaker who agreed (the recording page on virgoai.camksn.com makes them): add them as a dataset folder with `.wav` files and a `metadata.csv` (`file_name,sentence`) and set `EXTRA`. About 1–2 hours on a T4. It's uploaded to the same repo only if it finishes.

In [ ]:
# 4️⃣ LoRA fine-tune (optional)
EXTRA = ""       # optional: folder with .wav files + metadata.csv (file_name,sentence) from a speaker who agreed
ITERS = 1000
import csv, glob, yaml
SENTENCES = [
    "សួស្តី! តើអ្នកសុខសប្បាយជាទេ?", "ថ្ងៃនេះអាកាសធាតុល្អណាស់។", "ខ្ញុំអាចជួយអ្នកបកប្រែពីភាសាខ្មែរទៅភាសាអង់គ្លេសបាន។",
    "សូមអរគុណច្រើនសម្រាប់សំណួររបស់អ្នក។", "អង្គរវត្តគឺជាប្រាសាទដ៏ល្បីល្បាញបំផុតនៅកម្ពុជា។", "តើអ្នកចង់ដឹងអំពីអ្វីបន្ថែមទៀតទេ?",
    "ខ្ញុំនឹងពន្យល់ឱ្យងាយយល់។", "សូមរង់ចាំបន្តិច ខ្ញុំកំពុងរកមើលព័ត៌មាន។", "ភ្នំពេញជារាជធានីនៃព្រះរាជាណាចក្រកម្ពុជា។",
    "ការរៀនភាសាថ្មីត្រូវការពេលវេលា និងការអនុវត្តជាប្រចាំ។", "ខ្ញុំសូមណែនាំឱ្យអ្នកផឹកទឹកឱ្យបានច្រើន។", "តើថ្ងៃនេះអ្នកមានគម្រោងធ្វើអ្វីខ្លះ?",
    "Hello! I'm Virgo, your AI assistant.", "Let me explain that step by step.", "That's a great question.",
    "Here's a quick summary of what I found.", "Would you like me to translate this into Khmer?", "I'm happy to help with that.",
]
data = f"{WORK}/lora-data"; os.makedirs(data, exist_ok=True)
rows = []
if EXTRA:
    for r in csv.DictReader(open(os.path.join(EXTRA, "metadata.csv"), encoding="utf-8")):
        rows.append({"audio": os.path.join(EXTRA, r["file_name"]), "text": r["sentence"]})
else:
    voice = f"{WORK}/{MODEL_NAME}/voice.wav"
    for i, s in enumerate(SENTENCES * 4):  # 4 takes of each sentence
        wav = speak(s, prompt_wav_path=voice, prompt_text=SAMPLE_KM, reference_wav_path=voice,
                             cfg_value=2.0, inference_timesteps=10, seed=100 + i)
        path = f"{data}/{i:04}.wav"; sf.write(path, wav, RATE); rows.append({"audio": path, "text": s})
with open(f"{data}/train.jsonl", "w", encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(rows), "training clips")
del model; torch.cuda.empty_cache()
if not os.path.isdir(f"{WORK}/VoxCPM"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/OpenBMB/VoxCPM", f"{WORK}/VoxCPM"], check=True)
cfg = yaml.safe_load(open(f"{WORK}/VoxCPM/conf/voxcpm_v2/voxcpm_finetune_lora.yaml"))
cfg.update(pretrained_path=BASE, train_manifest=f"{data}/train.jsonl", batch_size=1, grad_accum_steps=16, num_workers=2,
           preprocessing_num_workers=2, num_iters=ITERS, max_steps=ITERS, save_interval=ITERS // 2,
           save_path=f"{WORK}/lora-out", tensorboard=f"{WORK}/lora-logs", hf_model_id="openbmb/VoxCPM2", distribute=True)
yaml.safe_dump(cfg, open(f"{WORK}/lora.yaml", "w"))
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
proc = subprocess.Popen([sys.executable, "scripts/train_voxcpm_finetune.py", "--config_path", f"{WORK}/lora.yaml"],
                        cwd=f"{WORK}/VoxCPM", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = []
for line in proc.stdout:
    tail = (tail + [line])[-40:]
    if "loss" in line.lower() or "error" in line.lower(): print(line, end="")
if proc.wait():
    print("".join(tail)); raise SystemExit("❌ Training stopped: send the lines above to Claude.")
latest = f"{WORK}/lora-out/latest"
for name in ("lora_weights.safetensors", "lora_config.json"):
    HfApi().upload_file(path_or_fileobj=f"{latest}/{name}", path_in_repo=name, repo_id=f"{ME}/{MODEL_NAME}")
print("☁️ LoRA uploaded to", f"{ME}/{MODEL_NAME}", "— restart the Virgo server notebook to use it.")